# DivergenceMagnetique — `divergence_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `divergence_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\nabla \cdot \mathbf{b} = \operatorname{tr}(\nabla b) = \sum_i \frac{\partial b_i}{\partial x_i}
$$
Exactly linear field $b = (x, y)$: $\nabla b = \mathbb{1}$, so $\nabla\cdot b = 2$ in
every cell (Green-Gauss is exact for linear fields).


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | magnetic field divergence | $\nabla\cdot b = \operatorname{tr}(\nabla b)$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/divergence_magnetique.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = 11                       # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 11, "ny": 11},            # the baseline mesh
    "standard": {"nx": 2*11-1, "ny": 2*11-1},
    "fine":     {"nx": 4*11-3, "ny": 4*11-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("divergence_magnetique.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition divergence_magnetique.data 2 -> PAR_divergence_magnetique.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
t, div = read_son(case.dataFileName_, "SONDE_DIV")

# b = (x, y) is exactly linear => div b = 1 + 1 = 2 at every cell centre.
expected = 2.0
print(f"divergence_magnetique (probe) : {div[-1]:.6g}  (expected {expected:.6g})")
err = abs(div[-1] - expected) / expected
print(f"relative error                : {err:.3e}")
assert err < 1e-12, f"divergence_magnetique wrong: {div[-1]}"
print("OK: divergence_magnetique = trace(grad b).")

# --- Boundary-cell check (B2: one-sided reconstruction) ---
t, div_bd = read_son(case.dataFileName_, "SONDE_DIV_BD")
print(f"divergence_magnetique (boundary): {div_bd[-1]:.6g}  (expected {expected:.6g})")
err_bd = abs(div_bd[-1] - expected) / expected
print(f"relative error (boundary)      : {err_bd:.3e}")
assert err_bd < 1e-12, f"boundary divergence_magnetique wrong: {div_bd[-1]}"
print("OK: boundary cell also recovers exact div b with B2 one-sided correction.")


## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`11x11`),
`standard` = cells x2 (`21x21`), `fine` = cells x4 (`41x41`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): b = (x,y) linear => div b = 2, constant. The scan asserts the same
machine-precision bound at **every** level
(the quantity is position-independent) — no order is reported.

In [ ]:
def read_final_dir(subdir, case_name, name):
    """(time, values) of probe `name` in a build subdir (final value = v[-1])."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    _, v = read_final_dir(subdir, case_name, "SONDE_DIV")
    expected = 2.0
    
    return abs(float(v[-1]) - float(expected)), expected

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (11, 11), "standard": (21, 21), "fine": (41, 41)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("coarse", "standard", "fine"):
    n0, n1 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e / max(abs(expected), 1e-300)
    print(f"{lvl:<9} {n0:>3}x{n1:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-12, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

